## MongoDB Code Generation Database Builder

This notebook is used to initialize a new code generation database onto MongoDB. A custom HumanEval dataset in csv format is processed into a set format through this notebook and stored in your MongoDB cluster.

In [2]:
import pandas as pd
import sys
import os

In [3]:
humaneval = pd.read_csv("/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/datasets/open_ended_format/humaneval_test_modified_open.csv", header = 0, encoding='unicode_escape')
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [ ]:
from code_generation.utility.humaneval_helper import CodeGenerationHumanEvalHelper
from database import MongoDBHelper

In [5]:
sample_qn = humaneval.iloc[32]
prompt = sample_qn['prompt']
canonical_solution = sample_qn['canonical_solution']
original_test = sample_qn['test']
print(prompt)

import math


def poly(xs: list, x: float):
    """
    Evaluates polynomial with coefficients xs at point x.
    return xs[0] + xs[1] * x + xs[1] * x^2 + .... xs[n] * x^n
    """
    return sum([coeff * math.pow(x, i) for i, coeff in enumerate(xs)])


def find_zero(xs: list):
    """ xs are coefficients of a polynomial.
    find_zero find x such that poly(x) = 0.
    find_zero returns only only zero point, even if there are many.
    Moreover, find_zero only takes list xs having even number of coefficients
    and largest non zero coefficient as it guarantees
    a solution.
    >>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x
    -0.5
    >>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3
    1.0
    """



In [6]:
new_qn, qn_desc = CodeGenerationHumanEvalHelper.seperate_original_desciptions(prompt)
print(new_qn)
print('-- qn_desc below --')
print(qn_desc)

import math

def poly(xs: list, x: float):
    """
    Evaluates polynomial with coefficients xs at point x.
    return xs[0] + xs[1] * x + xs[1] * x^2 + .... xs[n] * x^n
    """
    return sum([coeff * math.pow(x, i) for i, coeff in enumerate(xs)])

def find_zero(xs: list):
-- qn_desc below --
xs are coefficients of a polynomial.
find_zero find x such that poly(x) = 0.
find_zero returns only only zero point, even if there are many.
Moreover, find_zero only takes list xs having even number of coefficients
and largest non zero coefficient as it guarantees
a solution.
>>> round(find_zero([1, 2]), 2) # f(x) = 1 + 2x
-0.5
>>> round(find_zero([-6, 11, -6, 1]), 2) # (x - 1) * (x - 2) * (x - 3) = -6 + 11x - 6x^2 + x^3
1.0


In [8]:
desc, examples = CodeGenerationHumanEvalHelper.extract_examples(qn_desc)

In [12]:
check_function = CodeGenerationHumanEvalHelper.process_original_tests(original_test)
print(check_function)


def check(candidate):
    import math
    import random
    rng = random.Random(42)
    import copy
    for _ in range(100):
        ncoeff = 2 * rng.randint(1, 4)
        coeffs = []
        for _ in range(ncoeff):
            coeff = rng.randint(-10, 10)
            if coeff == 0:
                coeff = 1
            coeffs.append(coeff)
        solution = candidate(copy.deepcopy(coeffs))
        assert math.fabs(poly(coeffs, solution)) < 1e-4



## Sanity check to ensure the full solution passes check function

In [8]:
complete_solution = new_qn + "\n" + canonical_solution
check = CodeGenerationHumanEvalHelper.check_test_case(test_case=check_function, code_snippet = complete_solution, func_name="find_zero")

print("Function has passed the test" if check else "Function did not pass the test")

Function has passed the test


## Connecting to MongoDB

In [9]:
mongodbHelper = MongoDBHelper()
mongodbHelper.check_database_connectivity()

True

In [10]:
db = mongodbHelper.client["Base_Questions_DB"]
open_ended_db = db["HumanEval_Open_Ended"]

In [11]:
to_check_example = []
to_check_test = set()

for idx in range(humaneval.__len__()):

    qn_id = f"HumanEvalo{idx-len(to_check_example)-len(to_check_test)}"

    qn_details = open_ended_db.find_one({"_id" : qn_id})        # checking if this qn_id already exists in the db

    qn = humaneval.iloc[idx]
    prompt = qn['prompt']
    canonical_solution = qn['canonical_solution']
    tests = qn['test']
    original_test_id = qn['task_id']

    new_qn, qn_desc = CodeGenerationHumanEvalHelper.seperate_original_desciptions(prompt)

    desc, examples = CodeGenerationHumanEvalHelper.extract_examples(qn_desc)

    ## Rejecting any questions where there are no examples
    if len(examples.keys()) < 1:
        to_check_example.append(qn_id)
        continue

    ## Extracting function name
    first_example = list(examples.keys())[0]
    func_name = CodeGenerationHumanEvalHelper.extract_func_name_from_example(first_example)

    ## Cleaning the check function
    check_function = CodeGenerationHumanEvalHelper.process_original_tests(tests)

    ## Obtaining the full solution
    full_solution = new_qn + "\n" + canonical_solution

    ## Ensuring that the full solution passes the check function
    code_validation = CodeGenerationHumanEvalHelper.check_test_case(test_case = check_function, code_snippet = full_solution, func_name = func_name)
    
    entry_dict = {
        "_id" : qn_id,
        "qn" : new_qn,
        "canon_solution" : canonical_solution,
        "qn_desc" : desc,
        "examples": examples,
        "check" : check_function,
        "original_id": original_test_id
    }
    if code_validation is True:
        if qn_details is None:
            open_ended_db.insert_one(entry_dict)
            print('Added entry to database: {id}'.format(id = qn_id))
        else:
            open_ended_db.update_one({"_id" : qn_id}, update = {"$set": entry_dict})
            print('Updated existing entry in database: {id}'.format(id = qn_id))
    else:
        to_check_test.add(qn_id)

print(to_check_example if len(to_check_example) > 0 else "All cases contains examples. Nothing to check!")
print(to_check_test if len(to_check_test) > 0 else "All test cases passed. Nothing to check!")

Updated existing entry in database: HumanEvalo0
Updated existing entry in database: HumanEvalo1
Updated existing entry in database: HumanEvalo2
Updated existing entry in database: HumanEvalo3
Updated existing entry in database: HumanEvalo4
Updated existing entry in database: HumanEvalo5
Updated existing entry in database: HumanEvalo6
Updated existing entry in database: HumanEvalo7
Updated existing entry in database: HumanEvalo8
Updated existing entry in database: HumanEvalo9
Updated existing entry in database: HumanEvalo10
Updated existing entry in database: HumanEvalo11
Updated existing entry in database: HumanEvalo12
Updated existing entry in database: HumanEvalo13
Updated existing entry in database: HumanEvalo14
Updated existing entry in database: HumanEvalo15
Updated existing entry in database: HumanEvalo16
Updated existing entry in database: HumanEvalo17
Updated existing entry in database: HumanEvalo18
Updated existing entry in database: HumanEvalo19
Updated existing entry in data

Modified question HumanEval/10 to contain nested functions for uniformity.
HumanEval/41, HumanEval/38, HumanEval/50 did not have any examples in the quetions. 
HumanEval/66 to HumanEval/163 modified to fit the standard doctest format.